# **Rossmann Retail Store Sales Forecasting**


## 1. Project Overview

### 1.1 Project Objective

The objective is to analyze historical retail store data
and develop models for daily sales forecasting.

### 1.2 Notebook Scope

This notebook covers environment setup, project paths,
logging, dataset loading, initial data quality checks,
and the preparation required for validated data merging.

### 1.3 Data Integrity Principles

- Preserve all original CSV files.
- Document preprocessing decisions.
- Investigate missing values before handling them.
- Validate merge keys before joining datasets.
- Avoid data leakage during model preparation.

## 2. **Environment and Project Setup**

### 2.1 Import Libraries

Import the Python libraries required for filesystem
operations, data analysis, numerical calculations,
logging, and environment verification.

In [193]:
# Import basic libraries

from pathlib import Path
import logging
import sys

import pandas as pd
import numpy as np

# Display settings for easier data inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Notebook setup started successfully.")

Python version: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
Pandas version: 3.1.0
NumPy version: 2.5.4
Notebook setup started successfully.


### 2.2 **Verify Project and Dataset Paths**

Verify the notebook location, project root, raw data
directory, and the availability of all original CSV files.

In [194]:
# Verify project and dataset paths

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("Notebook location:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Raw data folder:", DATA_RAW)
print("\nCSV files found:")

for file_path in sorted(DATA_RAW.glob("*.csv")):
    print("-", file_path.name, "| Exists:", file_path.exists())

Notebook location: d:\Project_6_Rossmann_Forecasting\notebooks
Project root: d:\Project_6_Rossmann_Forecasting
Raw data folder: d:\Project_6_Rossmann_Forecasting\data\raw

CSV files found:
- sample_submission.csv | Exists: True
- store.csv | Exists: True
- test.csv | Exists: True
- train.csv | Exists: True


## 3. **Logging Configuration**

Configure Python logging to record project activities,
dataset loading, warnings, and errors in a log file
and the console.

In [195]:


import logging
from pathlib import Path

# Step 1: Define the project root directory.
# This notebook is expected to be inside the "notebooks" folder.
PROJECT_ROOT = Path.cwd().parent

# Step 2: Create the logs directory if it does not exist.
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Step 3: Define the log file path.
LOG_FILE = LOG_DIR / "project6.log"

# Step 4: Create a logger for Project 6.
logger = logging.getLogger("Project6")
logger.setLevel(logging.INFO)

# Step 5: Remove and close existing handlers.
# This prevents duplicate log messages when the cell is rerun.
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

# Prevent log messages from being duplicated by the root logger.
logger.propagate = False

# Step 6: Define the log message format.
log_format = logging.Formatter(
    fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Step 7: Create a file handler to save logs in project6.log.
file_handler = logging.FileHandler(
    LOG_FILE,
    mode="a",
    encoding="utf-8"
)
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(log_format)

# Step 8: Create a console handler to display logs in the notebook.
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(log_format)

# Step 9: Attach handlers to the logger.
logger.addHandler(file_handler)
logger.addHandler(console_handler)

# Step 10: Record the successful initialization.
logger.info("Project 6 logging system initialized successfully.")
logger.info("Log file location: %s", LOG_FILE.resolve())

# Step 11: Display confirmation.
print("Logging configured successfully.")
print(f"Log file: {LOG_FILE.resolve()}")


2026-10-26 11:41:52 | INFO | Project6 | Project 6 logging system initialized successfully.
2026-10-26 11:41:52 | INFO | Project6 | Log file location: D:\Project_6_Rossmann_Forecasting\logs\project6.log


Logging configured successfully.
Log file: D:\Project_6_Rossmann_Forecasting\logs\project6.log


## 4. **Dataset Loading**

Load the original train, test, store, and sample
submission CSV files without modifying the source data.

In [196]:

# Load the original CSV files
# Standardize StateHoliday during loading to avoid mixed types.

logger.info("Starting original dataset loading.")

train_df = pd.read_csv(
    DATA_RAW / "train.csv",
    dtype={"StateHoliday": "string"}
)

test_df = pd.read_csv(
    DATA_RAW / "test.csv",
    dtype={"StateHoliday": "string"}
)

store_df = pd.read_csv(DATA_RAW / "store.csv")

submission_df = pd.read_csv(
    DATA_RAW / "sample_submission.csv"
)

datasets = {
    "train": train_df,
    "test": test_df,
    "store": store_df,
    "sample_submission": submission_df,
}

for name, df in datasets.items():
    logger.info(
        "Loaded %s: %s rows, %s columns",
        name, df.shape[0], df.shape[1]
    )
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

# Verify StateHoliday has a consistent dtype.
print("\nTRAIN StateHoliday dtype:", train_df["StateHoliday"].dtype)
print("TEST StateHoliday dtype:", test_df["StateHoliday"].dtype)

print("\nTRAIN StateHoliday values:")
print(train_df["StateHoliday"].value_counts(dropna=False))

logger.info("All four CSV files loaded successfully.")


2026-10-26 11:41:52 | INFO | Project6 | Starting original dataset loading.
2026-10-26 11:41:52 | INFO | Project6 | Loaded train: 1017209 rows, 9 columns



TRAIN
Shape: (1017209, 9)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1


2026-10-26 11:41:52 | INFO | Project6 | Loaded test: 41088 rows, 8 columns



TEST
Shape: (41088, 8)
Columns: ['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
0,1,1,4,2015-09-17,1.0,1,0,0
1,2,3,4,2015-09-17,1.0,1,0,0
2,3,7,4,2015-09-17,1.0,1,0,0


... (4 outputs left out)


SAMPLE_SUBMISSION
Shape: (41088, 2)
Columns: ['Id', 'Sales']


,Id,Sales
0,1,0
1,2,0
2,3,0


2026-10-26 11:41:52 | INFO | Project6 | All four CSV files loaded successfully.



TRAIN StateHoliday dtype: string
TEST StateHoliday dtype: string

TRAIN StateHoliday values:
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: Int64


## 5. **Initial Data Understanding and Quality Audit**

### 5.1 Dataset Dimensions and Structure

### 5.2 First and Last Records

### 5.3 Column Names and Data Types

### 5.4 Descriptive Statistics

### 5.5 Missing Values and Unique Values

### 5.6 Initial Data Quality Observations

This section documents the structure, contents, statistical summaries, and initial quality of all four datasets before any data modification.

### 5.1 **Dataset Dimensions and Structure**

Inspect the number of rows and columns in each original
dataset before performing any preprocessing.

In [197]:
# Inspect the dimensions of all datasets

for name, df in datasets.items():
    print(f"{name.upper()}")
    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])
    print("-" * 40)

    logger.info(
        "%s dimensions: %d rows, %d columns",
        name, df.shape[0], df.shape[1]
    )

2026-10-26 11:41:53 | INFO | Project6 | train dimensions: 1017209 rows, 9 columns
2026-10-26 11:41:53 | INFO | Project6 | test dimensions: 41088 rows, 8 columns
2026-10-26 11:41:53 | INFO | Project6 | store dimensions: 1115 rows, 10 columns
2026-10-26 11:41:53 | INFO | Project6 | sample_submission dimensions: 41088 rows, 2 columns


TRAIN
Number of rows: 1017209
Number of columns: 9
----------------------------------------
TEST
Number of rows: 41088
Number of columns: 8
----------------------------------------
STORE
Number of rows: 1115
Number of columns: 10
----------------------------------------
SAMPLE_SUBMISSION
Number of rows: 41088
Number of columns: 2
----------------------------------------


### 5.2 **First and Last Records**

Inspect the first and last five records of each dataset
to understand the data layout and identify obvious issues.

In [198]:
# Inspect the first and last five rows

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"DATASET: {name.upper()}")
    print("=" * 50)

    print("\nFirst five rows:")
    display(df.head())

    print("\nLast five rows:")
    display(df.tail())

    logger.info(
        "Displayed first and last five rows for %s.",
        name
    )


DATASET: TRAIN

First five rows:


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1



Last five rows:


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
1017204,1111,2,2013-01-01,0,0,0,0,a,1
1017205,1112,2,2013-01-01,0,0,0,0,a,1
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1
1017208,1115,2,2013-01-01,0,0,0,0,a,1


2026-10-26 11:41:53 | INFO | Project6 | Displayed first and last five rows for train.



DATASET: TEST

First five rows:


... (10 outputs left out)

,Id,Sales
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0



Last five rows:


,Id,Sales
41083,41084,0
41084,41085,0
41085,41086,0
41086,41087,0
41087,41088,0


2026-10-26 11:41:53 | INFO | Project6 | Displayed first and last five rows for sample_submission.


### 5.3 **Column Names and Data Types**

Inspect all column names, data types, and non-null counts
to understand the schema of each dataset.

In [199]:
# Inspect schema and data types

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"SCHEMA: {name.upper()}")
    print("=" * 50)

    print("Column names:")
    print(df.columns.tolist())

    print("\nDataFrame information:")
    df.info()

    logger.info("Schema inspection completed for %s.", name)

2026-10-26 11:41:53 | INFO | Project6 | Schema inspection completed for train.
2026-10-26 11:41:53 | INFO | Project6 | Schema inspection completed for test.



SCHEMA: TRAIN
Column names:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   Store          1017209 non-null  int64 
 1   DayOfWeek      1017209 non-null  int64 
 2   Date           1017209 non-null  str   
 3   Sales          1017209 non-null  int64 
 4   Customers      1017209 non-null  int64 
 5   Open           1017209 non-null  int64 
 6   Promo          1017209 non-null  int64 
 7   StateHoliday   1017209 non-null  string
 8   SchoolHoliday  1017209 non-null  int64 
dtypes: int64(7), str(1), string(1)
memory usage: 69.8 MB

SCHEMA: TEST
Column names:
['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 41088 entrie

2026-10-26 11:41:53 | INFO | Project6 | Schema inspection completed for store.



SCHEMA: STORE
Column names:
['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str  

2026-10-26 11:41:53 | INFO | Project6 | Schema inspection completed for sample_submission.



SCHEMA: SAMPLE_SUBMISSION
Column names:
['Id', 'Sales']

DataFrame information:
<class 'pandas.DataFrame'>
RangeIndex: 41088 entries, 0 to 41087
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Id      41088 non-null  int64
 1   Sales   41088 non-null  int64
dtypes: int64(2)
memory usage: 642.1 KB


### 5.4 **Descriptive Statistics**

Review descriptive statistics to understand numeric
ranges and the distribution of values in each dataset.

In [200]:
#  Generate descriptive statistics

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"DESCRIPTIVE STATISTICS: {name.upper()}")
    print("=" * 50)

    display(df.describe(include="all").T)

    logger.info(
        "Descriptive statistics generated for %s.", name
    )


DESCRIPTIVE STATISTICS: TRAIN


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Store,1017209.0,NaN,NaN,NaN,558.429727,321.908651,1.0,280.0,558.0,838.0,1115.0
DayOfWeek,1017209.0,NaN,NaN,NaN,3.998341,1.997391,1.0,2.0,4.0,6.0,7.0
Date,1017209,942,2015-07-31,1115,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sales,1017209.0,NaN,NaN,NaN,5773.818972,3849.926175,0.0,3727.0,5744.0,7856.0,41551.0
Customers,1017209.0,NaN,NaN,NaN,633.145946,464.411734,0.0,405.0,609.0,837.0,7388.0
Open,1017209.0,NaN,NaN,NaN,0.830107,0.375539,0.0,1.0,1.0,1.0,1.0
Promo,1017209.0,NaN,NaN,NaN,0.381515,0.485759,0.0,0.0,0.0,1.0,1.0
StateHoliday,1017209,4,0,986159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SchoolHoliday,1017209.0,NaN,NaN,NaN,0.178647,0.383056,0.0,0.0,0.0,0.0,1.0


2026-10-26 11:41:53 | INFO | Project6 | Descriptive statistics generated for train.



DESCRIPTIVE STATISTICS: TEST


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Id,41088.0,NaN,NaN,NaN,20544.5,11861.228267,1.0,10272.75,20544.5,30816.25,41088.0
Store,41088.0,NaN,NaN,NaN,555.899533,320.274496,1.0,279.75,553.5,832.25,1115.0
DayOfWeek,41088.0,NaN,NaN,NaN,3.979167,2.015481,1.0,2.0,4.0,6.0,7.0
Date,41088,48,2015-09-17,856,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Open,41077.0,NaN,NaN,NaN,0.854322,0.352787,0.0,1.0,1.0,1.0,1.0
Promo,41088.0,NaN,NaN,NaN,0.395833,0.489035,0.0,0.0,0.0,1.0,1.0
StateHoliday,41088,2,0,40908,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SchoolHoliday,41088.0,NaN,NaN,NaN,0.443487,0.496802,0.0,0.0,0.0,1.0,1.0


2026-10-26 11:41:53 | INFO | Project6 | Descriptive statistics generated for test.


... (2 outputs left out)

2026-10-26 11:41:53 | INFO | Project6 | Descriptive statistics generated for store.



DESCRIPTIVE STATISTICS: SAMPLE_SUBMISSION


,count,mean,std,min,25%,50%,75%,max
Id,41088.0,20544.5,11861.228267,1.0,10272.75,20544.5,30816.25,41088.0
Sales,41088.0,0.0,0.000000,0.0,0.00,0.0,0.00,0.0


2026-10-26 11:41:53 | INFO | Project6 | Descriptive statistics generated for sample_submission.


### 5.5 **Missing Values and Unique Values**

This section quantifies missing values and examines the
number of distinct non-null values in each column.
The results will guide later preprocessing decisions.
No imputation or deletion is performed in this section.

In [201]:
# Missing-value and unique-value audit

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"MISSING VALUES AND UNIQUE VALUES: {name.upper()}")
    print("=" * 60)

    audit = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(dropna=True)
    })

    display(audit)

    logger.info(
        "Missing-value and unique-value audit completed for %s.",
        name
    )


MISSING VALUES AND UNIQUE VALUES: TRAIN


,data_type,missing_count,missing_percent,unique_values
Store,int64,0,0.0,1115
DayOfWeek,int64,0,0.0,7
Date,str,0,0.0,942
Sales,int64,0,0.0,21734
Customers,int64,0,0.0,4086
Open,int64,0,0.0,2
Promo,int64,0,0.0,2
StateHoliday,string,0,0.0,4
SchoolHoliday,int64,0,0.0,2


2026-10-26 11:41:53 | INFO | Project6 | Missing-value and unique-value audit completed for train.



MISSING VALUES AND UNIQUE VALUES: TEST


,data_type,missing_count,missing_percent,unique_values
Id,int64,0,0.00,41088
Store,int64,0,0.00,856
DayOfWeek,int64,0,0.00,7
Date,str,0,0.00,48
Open,float64,11,0.03,2
Promo,int64,0,0.00,2
StateHoliday,string,0,0.00,2
SchoolHoliday,int64,0,0.00,2


2026-10-26 11:41:53 | INFO | Project6 | Missing-value and unique-value audit completed for test.


... (2 outputs left out)

2026-10-26 11:41:53 | INFO | Project6 | Missing-value and unique-value audit completed for store.



MISSING VALUES AND UNIQUE VALUES: SAMPLE_SUBMISSION


,data_type,missing_count,missing_percent,unique_values
Id,int64,0,0.0,41088
Sales,int64,0,0.0,1


2026-10-26 11:41:53 | INFO | Project6 | Missing-value and unique-value audit completed for sample_submission.



### 5.6 **Initial Data Quality Observations**

The initial audit identified the following observations:

1. The training dataset has no missing values in the displayed audit results.
2. The test dataset contains 11 missing values in the `Open` column.
3. The store metadata contains missing values in competition-related and `Promo2`-related columns.
4. The `sample_submission.csv` file contains zero values in its `Sales` column. These are placeholder values, not observed sales.
5. The `StateHoliday` column is loaded using the pandas `string` dtype to avoid mixed-type inference during CSV loading. Its observed categories are inspected in the validation section.

No missing values have been imputed or removed at this stage.


## 6. **Data Validation**

This section checks the integrity and consistency of the datasets before preprocessing and merging.

### 6.1 **Duplicate Records**


In [202]:
# Section 6.1: Check exact duplicate rows in each dataset

for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print(f"\nDataset: {name}")
    print(f"Exact duplicate rows: {duplicate_count}")

    logger.info(
        "%s: found %d exact duplicate rows.",
        name,
        duplicate_count
    )

2026-10-26 11:41:53 | INFO | Project6 | train: found 0 exact duplicate rows.
2026-10-26 11:41:53 | INFO | Project6 | test: found 0 exact duplicate rows.
2026-10-26 11:41:53 | INFO | Project6 | store: found 0 exact duplicate rows.
2026-10-26 11:41:53 | INFO | Project6 | sample_submission: found 0 exact duplicate rows.



Dataset: train
Exact duplicate rows: 0

Dataset: test
Exact duplicate rows: 0

Dataset: store
Exact duplicate rows: 0

Dataset: sample_submission
Exact duplicate rows: 0


### 6.2 **Date Range and Date Validity**

Inspect the date columns in the training and test datasets.
Check the minimum and maximum dates, invalid dates,
and whether the data covers the expected time periods.

In [203]:
# Validate dates in train and test datasets

for name, df in {"train": train_df, "test": test_df}.items():
    print(f"\n{'=' * 50}")
    print(f"DATE VALIDATION: {name.upper()}")
    print("=" * 50)

    # Convert Date to datetime in a separate Series.
    # Original dataframe is not modified in this check.
    parsed_dates = pd.to_datetime(df["Date"], errors="coerce")

    invalid_count = parsed_dates.isna().sum()

    print("Rows:", len(df))
    print("Invalid or missing dates:", invalid_count)
    print("Earliest date:", parsed_dates.min())
    print("Latest date:", parsed_dates.max())
    print("Unique dates:", parsed_dates.nunique())

    logger.info(
        "%s date validation: invalid=%d, unique_dates=%d",
        name,
        invalid_count,
        parsed_dates.nunique()
    )

2026-10-26 11:41:54 | INFO | Project6 | train date validation: invalid=0, unique_dates=942
2026-10-26 11:41:54 | INFO | Project6 | test date validation: invalid=0, unique_dates=48



DATE VALIDATION: TRAIN
Rows: 1017209
Invalid or missing dates: 0
Earliest date: 2013-01-01 00:00:00
Latest date: 2015-07-31 00:00:00
Unique dates: 942

DATE VALIDATION: TEST
Rows: 41088
Invalid or missing dates: 0
Earliest date: 2015-08-01 00:00:00
Latest date: 2015-09-17 00:00:00
Unique dates: 48


### 6.3 **Key and Column Validation**

Validate the required columns, store identifiers, and key uniqueness before merging the store metadata with the training and test datasets.

In [204]:
# Validate required columns in each dataset

required_columns = {
    "train": {
        "Store", "Date", "Sales", "Customers",
        "Open", "Promo", "StateHoliday", "SchoolHoliday"
    },
    "test": {
        "Id", "Store", "Date", "Open",
        "Promo", "StateHoliday", "SchoolHoliday"
    },
    "store": {
        "Store", "StoreType", "Assortment",
        "CompetitionDistance", "Promo2"
    },
    "sample_submission": {"Id", "Sales"}
}

for name, df in datasets.items():
    required = required_columns[name]
    missing_columns = sorted(required - set(df.columns))

    print(f"\nDataset: {name}")
    print("Required columns:", len(required))
    print("Missing required columns:", missing_columns)

    if missing_columns:
        logger.warning(
            "%s is missing required columns: %s",
            name, missing_columns
        )
    else:
        logger.info(
            "%s contains all checked required columns.",
            name
        )

2026-10-26 11:41:54 | INFO | Project6 | train contains all checked required columns.
2026-10-26 11:41:54 | INFO | Project6 | test contains all checked required columns.
2026-10-26 11:41:54 | INFO | Project6 | store contains all checked required columns.
2026-10-26 11:41:54 | INFO | Project6 | sample_submission contains all checked required columns.



Dataset: train
Required columns: 8
Missing required columns: []

Dataset: test
Required columns: 7
Missing required columns: []

Dataset: store
Required columns: 5
Missing required columns: []

Dataset: sample_submission
Required columns: 2
Missing required columns: []


In [205]:
# Validate Store IDs and merge-key integrity

# Check whether each Store appears only once in store.csv
duplicate_store_ids = store_df["Store"].duplicated().sum()

print("Duplicate Store IDs in store.csv:", duplicate_store_ids)
print("Unique Store IDs in store.csv:", store_df["Store"].nunique())

# Compare train and test store IDs against store metadata
metadata_store_ids = set(store_df["Store"].dropna().unique())
train_store_ids = set(train_df["Store"].dropna().unique())
test_store_ids = set(test_df["Store"].dropna().unique())

print(
    "Train Store IDs missing from metadata:",
    len(train_store_ids - metadata_store_ids)
)
print(
    "Test Store IDs missing from metadata:",
    len(test_store_ids - metadata_store_ids)
)

logger.info(
    "Store-key validation: duplicate metadata keys=%d",
    duplicate_store_ids
)

2026-10-26 11:41:54 | INFO | Project6 | Store-key validation: duplicate metadata keys=0


Duplicate Store IDs in store.csv: 0
Unique Store IDs in store.csv: 1115
Train Store IDs missing from metadata: 0
Test Store IDs missing from metadata: 0


### 6.4 **Sample Submission Validation**

Inspect the sample submission template to verify its structure, identifier uniqueness, target column, and placeholder values. This file defines the expected format of the final sales predictions.

In [206]:
# Validate the sample submission template

print("Sample submission shape:", submission_df.shape)
print("\nFirst five rows:")
display(submission_df.head())

print("\nColumn names:", submission_df.columns.tolist())
print("Unique IDs:", submission_df["Id"].nunique())
print("Duplicate IDs:", submission_df["Id"].duplicated().sum())
print("Missing IDs:", submission_df["Id"].isna().sum())
print("Missing Sales:", submission_df["Sales"].isna().sum())
print("Unique placeholder Sales values:",
      submission_df["Sales"].unique())

# Compare submission IDs with test IDs
print(
    "\nSubmission IDs match test IDs:",
    submission_df["Id"].equals(test_df["Id"])
)

logger.info("Sample submission template validation completed.")

Sample submission shape: (41088, 2)

First five rows:


,Id,Sales
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


2026-10-26 11:41:54 | INFO | Project6 | Sample submission template validation completed.



Column names: ['Id', 'Sales']
Unique IDs: 41088
Duplicate IDs: 0
Missing IDs: 0
Missing Sales: 0
Unique placeholder Sales values: [0]

Submission IDs match test IDs: True


### 6.5 **Categorical Value Inspection**

Inspect the unique values and frequencies of categorical
columns to identify inconsistent representations before
preprocessing.

This step focuses on StateHoliday, StoreType, Assortment,
Promo, Promo2, and SchoolHoliday.

In [207]:
# Inspect categorical values

categorical_columns = {
    "train": [
        "StateHoliday", "Promo", "SchoolHoliday", "Open"
    ],
    "test": [
        "StateHoliday", "Promo", "SchoolHoliday", "Open"
    ],
    "store": [
        "StoreType", "Assortment", "Promo2", "PromoInterval"
    ]
}

for name, columns in categorical_columns.items():
    df = datasets[name]

    print(f"\n{'=' * 60}")
    print(f"CATEGORICAL VALUE INSPECTION: {name.upper()}")
    print("=" * 60)

    for column in columns:
        if column in df.columns:
            print(f"\nColumn: {column}")
            display(
                df[column]
                .value_counts(dropna=False)
                .rename_axis(column)
                .reset_index(name="count")
            )

            logger.info(
                "Inspected categorical values: %s.%s",
                name, column
            )


CATEGORICAL VALUE INSPECTION: TRAIN

Column: StateHoliday


,StateHoliday,count
0,0,986159
1,a,20260
2,b,6690
3,c,4100


2026-10-26 11:41:54 | INFO | Project6 | Inspected categorical values: train.StateHoliday



Column: Promo


,Promo,count
0,0,629129
1,1,388080


2026-10-26 11:41:54 | INFO | Project6 | Inspected categorical values: train.Promo


... (26 outputs left out)

2026-10-26 11:41:54 | INFO | Project6 | Inspected categorical values: store.Promo2



Column: PromoInterval


,PromoInterval,count
0,NaN,544
1,"Jan,Apr,Jul,Oct",335
2,"Feb,May,Aug,Nov",130
3,"Mar,Jun,Sept,Dec",106


2026-10-26 11:41:54 | INFO | Project6 | Inspected categorical values: store.PromoInterval


### 6.6 **Row-Level Key Validation**

Check the uniqueness of the Store–Date combination in the training and test datasets. Validate the test identifier column separately to ensure that records can be matched reliably during prediction and submission generation.

In [208]:
# Validate row-level keys

for name, df in {"train": train_df, "test": test_df}.items():
    duplicate_store_date = df.duplicated(
        subset=["Store", "Date"]
    ).sum()

    print(f"\nDataset: {name.upper()}")
    print("Duplicate Store-Date pairs:", duplicate_store_date)

    logger.info(
        "%s duplicate Store-Date pairs: %d",
        name, duplicate_store_date
    )

# Test IDs should uniquely identify test records
print("\nTest ID checks")
print("Missing IDs:", test_df["Id"].isna().sum())
print("Duplicate IDs:", test_df["Id"].duplicated().sum())

logger.info("Row-level key validation completed.")


Dataset: TRAIN
Duplicate Store-Date pairs: 0


2026-10-26 11:41:54 | INFO | Project6 | train duplicate Store-Date pairs: 0
2026-10-26 11:41:54 | INFO | Project6 | test duplicate Store-Date pairs: 0
2026-10-26 11:41:54 | INFO | Project6 | Row-level key validation completed.



Dataset: TEST
Duplicate Store-Date pairs: 0

Test ID checks
Missing IDs: 0
Duplicate IDs: 0



### 6.7 **StateHoliday Data-Type and Category Validation**

The `StateHoliday` column is loaded with pandas' `string` dtype to prevent mixed-type inference during CSV loading. This section verifies the observed category values and checks whether any missing or unexpected categories are present.

No source data will be modified during this investigation.


In [209]:
# Investigate StateHoliday representations

for name, df in {"train": train_df, "test": test_df}.items():
    print(f"\n{'=' * 55}")
    print(f"STATEHOLIDAY TYPE INVESTIGATION: {name.upper()}")
    print("=" * 55)

    values = df["StateHoliday"].dropna().unique()

    for value in values:
        print(
            f"Value: {repr(value):>6} | "
            f"Python type: {type(value).__name__}"
        )

    print("\nValue counts with representations:")
    counts = df["StateHoliday"].value_counts(dropna=False)

    for value, count in counts.items():
        print(
            f"Value: {repr(value):>6} | Count: {count}"
        )

    logger.info(
        "StateHoliday representation investigation completed for %s.",
        name
    )


STATEHOLIDAY TYPE INVESTIGATION: TRAIN
Value:    '0' | Python type: str
Value:    'a' | Python type: str
Value:    'b' | Python type: str
Value:    'c' | Python type: str

Value counts with representations:
Value:    '0' | Count: 986159
Value:    'a' | Count: 20260
Value:    'b' | Count: 6690
Value:    'c' | Count: 4100


2026-10-26 11:41:54 | INFO | Project6 | StateHoliday representation investigation completed for train.



STATEHOLIDAY TYPE INVESTIGATION: TEST


2026-10-26 11:41:54 | INFO | Project6 | StateHoliday representation investigation completed for test.


Value:    '0' | Python type: str
Value:    'a' | Python type: str

Value counts with representations:
Value:    '0' | Count: 40908
Value:    'a' | Count: 180



### 6.8 **Data Validation Summary**

This section summarizes the validation results before preprocessing. The checks cover exact duplicate records, date validity, required columns, store metadata keys, sample submission structure, categorical values, and row-level key integrity.

The audit identified missing values in selected test and store metadata columns. The `StateHoliday` column is loaded as a string to maintain consistent categorical representation. The `Sales` values in the sample submission are placeholders and must not be treated as actual sales observations.

The validation checks are calculated from the loaded datasets. Any failed check must be investigated before merging.


In [210]:

#  Dynamic Data Validation Summary

validation_rows = []

# 1. Exact duplicate records
for name, df in datasets.items():
    validation_rows.append({
        "Check": f"{name}: exact duplicate rows",
        "Result": int(df.duplicated().sum()),
        "Expected": 0
    })

# 2. Date validity
for name, df in {"train": train_df, "test": test_df}.items():
    parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
    validation_rows.append({
        "Check": f"{name}: invalid or missing dates",
        "Result": int(parsed_dates.isna().sum()),
        "Expected": 0
    })

# 3. Store and row-level key checks
validation_rows.extend([
    {
        "Check": "Duplicate Store IDs in store metadata",
        "Result": int(store_df["Store"].duplicated().sum()),
        "Expected": 0
    },
    {
        "Check": "Train Store IDs missing from metadata",
        "Result": len(set(train_df["Store"]) - set(store_df["Store"])),
        "Expected": 0
    },
    {
        "Check": "Test Store IDs missing from metadata",
        "Result": len(set(test_df["Store"]) - set(store_df["Store"])),
        "Expected": 0
    },
    {
        "Check": "Duplicate train Store-Date pairs",
        "Result": int(train_df.duplicated(["Store", "Date"]).sum()),
        "Expected": 0
    },
    {
        "Check": "Duplicate test Store-Date pairs",
        "Result": int(test_df.duplicated(["Store", "Date"]).sum()),
        "Expected": 0
    },
    {
        "Check": "Duplicate test IDs",
        "Result": int(test_df["Id"].duplicated().sum()),
        "Expected": 0
    },
    {
        "Check": "Submission IDs match test IDs in order",
        "Result": bool(submission_df["Id"].equals(test_df["Id"])),
        "Expected": True
    }
])

# 4. Required-column checks
required_columns = {
    "train": {
        "Store", "DayOfWeek", "Date", "Sales", "Customers",
        "Open", "Promo", "StateHoliday", "SchoolHoliday"
    },
    "test": {
        "Id", "Store", "DayOfWeek", "Date", "Open",
        "Promo", "StateHoliday", "SchoolHoliday"
    },
    "store": {
        "Store", "StoreType", "Assortment", "CompetitionDistance",
        "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
        "Promo2", "Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"
    },
    "sample_submission": {"Id", "Sales"}
}

for name, expected_columns in required_columns.items():
    missing_columns = expected_columns - set(datasets[name].columns)
    validation_rows.append({
        "Check": f"{name}: required columns present",
        "Result": len(missing_columns),
        "Expected": 0
    })

# 5. Create the validation report
validation_summary = pd.DataFrame(validation_rows)
validation_summary["Passed"] = (
    validation_summary["Result"] == validation_summary["Expected"]
)

display(validation_summary)

failed_checks = validation_summary.loc[
    ~validation_summary["Passed"]
]

if failed_checks.empty:
    logger.info("All dynamic data validation checks passed.")
    print("PASS: All validation checks passed.")
else:
    logger.error("%d validation checks failed.", len(failed_checks))
    print("WARNING: Investigate the failed checks below.")
    display(failed_checks)


,Check,Result,Expected,Passed
0,train: exact duplicate rows,0,0,True
1,test: exact duplicate rows,0,0,True
2,store: exact duplicate rows,0,0,True
3,sample_submission: exact duplicate rows,0,0,True
4,train: invalid or missing dates,0,0,True
5,test: invalid or missing dates,0,0,True
6,Duplicate Store IDs in store metadata,0,0,True
7,Train Store IDs missing from metadata,0,0,True
8,Test Store IDs missing from metadata,0,0,True
9,Duplicate train Store-Date pairs,0,0,True


2026-10-26 11:41:54 | INFO | Project6 | All dynamic data validation checks passed.


PASS: All validation checks passed.


## 7. **Data Dictionary**

This section documents the columns, data types, meanings, and analytical uses of the original datasets. The dictionary will serve as a reference for preprocessing, feature engineering, model development, and final prediction generation.

### 7.1 Training Dataset

### 7.2 Test Dataset

### 7.3 Store Metadata

### 7.4 Sample Submission

### 7.1 **Training Dataset — Data Dictionary**

This section documents the columns in `train.csv`, including their observed data types, business meanings, and roles in sales forecasting. The training dataset contains historical store-level daily records and the target variable `Sales`.

Column descriptions are based on the project dataset and its documented business context. Observed data types will be verified against the loaded dataframe.

In [211]:
# Data Dictionary for the training dataset

train_descriptions = {
    "Store": (
        "Unique identifier of a retail store.",
        "Store-level analysis and joining store metadata."
    ),
    "DayOfWeek": (
        "Day of the week, represented by an integer from 1 to 7.",
        "Weekly sales and customer behavior analysis."
    ),
    "Date": (
        "Date of the recorded store transaction activity.",
        "Time-based analysis and forecasting features."
    ),
    "Sales": (
        "Daily store turnover; the main prediction target.",
        "Target variable for sales forecasting."
    ),
    "Customers": (
        "Number of customers recorded for the store on that day.",
        "Customer behavior analysis; use carefully to avoid target leakage."
    ),
    "Open": (
        "Indicates whether the store was open: 0 = closed, 1 = open.",
        "Store operating-status analysis and forecasting."
    ),
    "Promo": (
        "Indicates whether a promotion was active on that day.",
        "Promotion impact analysis and forecasting."
    ),
    "StateHoliday": (
        "State holiday category: 0 = none, a = public holiday, "
        "b = Easter holiday, c = Christmas.",
        "Holiday-related sales analysis and forecasting."
    ),
    "SchoolHoliday": (
        "Indicates whether the date was affected by school holidays.",
        "School-holiday behavior analysis and forecasting."
    )
}

dictionary_rows = []

for column in train_df.columns:
    description, project_use = train_descriptions.get(
        column,
        (
            "Description requires verification.",
            "Determine after reviewing the dataset documentation."
        )
    )

    dictionary_rows.append({
        "Column Name": column,
        "Observed Data Type": str(train_df[column].dtype),
        "Meaning / Description": description,
        "Project Use": project_use
    })

train_data_dictionary = pd.DataFrame(dictionary_rows)

display(train_data_dictionary)

logger.info(
    "Training dataset data dictionary generated for %d columns.",
    len(train_data_dictionary)
)

,Column Name,Observed Data Type,Meaning / Description,Project Use
0,Store,int64,Unique identifier of a retail store.,Store-level analysis and joining store metadata.
1,DayOfWeek,int64,"Day of the week, represented by an integer fro...",Weekly sales and customer behavior analysis.
2,Date,str,Date of the recorded store transaction activity.,Time-based analysis and forecasting features.
3,Sales,int64,Daily store turnover; the main prediction target.,Target variable for sales forecasting.
4,Customers,int64,Number of customers recorded for the store on ...,Customer behavior analysis; use carefully to a...
5,Open,int64,Indicates whether the store was open: 0 = clos...,Store operating-status analysis and forecasting.
6,Promo,int64,Indicates whether a promotion was active on th...,Promotion impact analysis and forecasting.
7,StateHoliday,string,"State holiday category: 0 = none, a = public h...",Holiday-related sales analysis and forecasting.
8,SchoolHoliday,int64,Indicates whether the date was affected by sch...,School-holiday behavior analysis and forecasting.


2026-10-26 11:41:54 | INFO | Project6 | Training dataset data dictionary generated for 9 columns.


### 7.2 **Test Dataset — Data Dictionary**

This section documents the columns in `test.csv`, including their observed data types, meanings, and roles in forecasting. Unlike the training dataset, the test dataset does not contain the actual Sales target values. Its `Id` column identifies each test record and must be preserved for the final prediction submission.

In [212]:
# Data Dictionary for the test dataset

test_descriptions = {
    "Id": (
        "Unique identifier for a Store-Date record in the test dataset.",
        "Preserve this identifier and match predictions to the submission template."
    ),
    "Store": (
        "Unique identifier of the retail store.",
        "Join store metadata and identify store-level patterns."
    ),
    "DayOfWeek": (
        "Day of the week, represented by an integer from 1 to 7.",
        "Weekly sales behavior and forecasting features."
    ),
    "Date": (
        "Date for which sales need to be predicted.",
        "Chronological validation and date-based feature engineering."
    ),
    "Open": (
        "Indicates whether the store was open: 0 = closed, 1 = open.",
        "Store operating status; investigate missing values before treatment."
    ),
    "Promo": (
        "Indicates whether a promotion was active on that day.",
        "Promotion analysis and sales forecasting."
    ),
    "StateHoliday": (
        "State holiday category: 0 = none, a = public holiday, "
        "b = Easter holiday, c = Christmas.",
        "Holiday-related analysis and forecasting."
    ),
    "SchoolHoliday": (
        "Indicates whether the date was affected by school holidays.",
        "School-holiday analysis and forecasting."
    )
}

test_dictionary_rows = []

for column in test_df.columns:
    description, project_use = test_descriptions.get(
        column,
        (
            "Description requires verification.",
            "Determine after reviewing the dataset documentation."
        )
    )

    test_dictionary_rows.append({
        "Column Name": column,
        "Observed Data Type": str(test_df[column].dtype),
        "Meaning / Description": description,
        "Project Use": project_use
    })

test_data_dictionary = pd.DataFrame(test_dictionary_rows)

display(test_data_dictionary)

logger.info(
    "Test dataset data dictionary generated for %d columns.",
    len(test_data_dictionary)
)

,Column Name,Observed Data Type,Meaning / Description,Project Use
0,Id,int64,Unique identifier for a Store-Date record in t...,Preserve this identifier and match predictions...
1,Store,int64,Unique identifier of the retail store.,Join store metadata and identify store-level p...
2,DayOfWeek,int64,"Day of the week, represented by an integer fro...",Weekly sales behavior and forecasting features.
3,Date,str,Date for which sales need to be predicted.,Chronological validation and date-based featur...
4,Open,float64,Indicates whether the store was open: 0 = clos...,Store operating status; investigate missing va...
5,Promo,int64,Indicates whether a promotion was active on th...,Promotion analysis and sales forecasting.
6,StateHoliday,string,"State holiday category: 0 = none, a = public h...",Holiday-related analysis and forecasting.
7,SchoolHoliday,int64,Indicates whether the date was affected by sch...,School-holiday analysis and forecasting.


2026-10-26 11:41:54 | INFO | Project6 | Test dataset data dictionary generated for 8 columns.


### 7.3 **Store Metadata — Data Dictionary**

This section documents the store-level attributes provided in `store.csv`. These attributes describe store characteristics, assortment, competition, and promotional programs. They will be used to support exploratory data analysis and forecasting after the metadata has been validated and merged with the training and test datasets.

In [213]:
#  Data Dictionary for store metadata

store_descriptions = {
    "Store": (
        "Unique identifier of a retail store.",
        "Join key for combining store metadata with train and test records."
    ),
    "StoreType": (
        "Categorical classification of the store.",
        "Compare sales patterns across store types."
    ),
    "Assortment": (
        "Level or type of product assortment offered by the store.",
        "Analyze the relationship between assortment and sales."
    ),
    "CompetitionDistance": (
        "Approximate distance in meters to the nearest competitor store.",
        "Study competition proximity and its relationship with sales."
    ),
    "CompetitionOpenSinceMonth": (
        "Month when the nearest competitor opened, where recorded.",
        "Construct competition-duration features when valid."
    ),
    "CompetitionOpenSinceYear": (
        "Year when the nearest competitor opened, where recorded.",
        "Construct competition-duration features when valid."
    ),
    "Promo2": (
        "Indicates whether the store participates in the continuing Promo2 program.",
        "Analyze long-running promotion participation."
    ),
    "Promo2SinceWeek": (
        "Calendar week when Promo2 began for the store, where recorded.",
        "Derive promotion timing features when valid."
    ),
    "Promo2SinceYear": (
        "Year when Promo2 began for the store, where recorded.",
        "Derive promotion timing features when valid."
    ),
    "PromoInterval": (
        "Months in which the store's Promo2 promotion is repeated.",
        "Determine whether Promo2 is active for a given date."
    )
}

store_dictionary_rows = []

for column in store_df.columns:
    description, project_use = store_descriptions.get(
        column,
        (
            "Description requires verification.",
            "Determine after reviewing the dataset documentation."
        )
    )

    store_dictionary_rows.append({
        "Column Name": column,
        "Observed Data Type": str(store_df[column].dtype),
        "Meaning / Description": description,
        "Project Use": project_use
    })

store_data_dictionary = pd.DataFrame(store_dictionary_rows)

display(store_data_dictionary)

logger.info(
    "Store metadata data dictionary generated for %d columns.",
    len(store_data_dictionary)
)

,Column Name,Observed Data Type,Meaning / Description,Project Use
0,Store,int64,Unique identifier of a retail store.,Join key for combining store metadata with tra...
1,StoreType,str,Categorical classification of the store.,Compare sales patterns across store types.
2,Assortment,str,Level or type of product assortment offered by...,Analyze the relationship between assortment an...
3,CompetitionDistance,float64,Approximate distance in meters to the nearest ...,Study competition proximity and its relationsh...
4,CompetitionOpenSinceMonth,float64,"Month when the nearest competitor opened, wher...",Construct competition-duration features when v...
5,CompetitionOpenSinceYear,float64,"Year when the nearest competitor opened, where...",Construct competition-duration features when v...
6,Promo2,int64,Indicates whether the store participates in th...,Analyze long-running promotion participation.
7,Promo2SinceWeek,float64,"Calendar week when Promo2 began for the store,...",Derive promotion timing features when valid.
8,Promo2SinceYear,float64,"Year when Promo2 began for the store, where re...",Derive promotion timing features when valid.
9,PromoInterval,str,Months in which the store's Promo2 promotion i...,Determine whether Promo2 is active for a given...


2026-10-26 11:41:54 | INFO | Project6 | Store metadata data dictionary generated for 10 columns.


### 7.4 **Sample Submission — Data Dictionary**

This section documents the columns in `sample_submission.csv`. The file provides the required output structure for submitting sales predictions. Its Sales column contains placeholder values and must be populated with model predictions for the corresponding test IDs only when generating the final submission.

In [214]:
# Data Dictionary for sample submission

submission_descriptions = {
    "Id": (
        "Identifier corresponding to a test dataset record.",
        "Match each predicted value to the correct test record."
    ),
    "Sales": (
        "Placeholder column for predicted daily sales.",
        "Store final model predictions in the required submission format."
    )
}

submission_dictionary_rows = []

for column in submission_df.columns:
    description, project_use = submission_descriptions.get(
        column,
        (
            "Description requires verification.",
            "Determine after reviewing the submission requirements."
        )
    )

    submission_dictionary_rows.append({
        "Column Name": column,
        "Observed Data Type": str(submission_df[column].dtype),
        "Meaning / Description": description,
        "Project Use": project_use
    })

submission_data_dictionary = pd.DataFrame(
    submission_dictionary_rows
)

display(submission_data_dictionary)

logger.info(
    "Sample submission data dictionary generated for %d columns.",
    len(submission_data_dictionary)
)

,Column Name,Observed Data Type,Meaning / Description,Project Use
0,Id,int64,Identifier corresponding to a test dataset rec...,Match each predicted value to the correct test...
1,Sales,int64,Placeholder column for predicted daily sales.,Store final model predictions in the required ...


2026-10-26 11:41:54 | INFO | Project6 | Sample submission data dictionary generated for 2 columns.


## 8. **Data Preprocessing and Merging**

This section prepares the original datasets for analysis and forecasting. It will address data-type consistency, missing values, key integrity, and store metadata integration. Original source files will remain unchanged, and each transformation will be logged and validated.

### 8.1 **Preprocessing Strategy and Source Preservation**

Before modifying any data, preserve the original dataframes and define separate working copies. All preprocessing decisions will be based on the preceding validation results. Row counts, column names, keys, and missing-value counts will be checked after relevant transformations.

In [215]:
# Create working copies of the source datasets

train_work = train_df.copy(deep=True)
test_work = test_df.copy(deep=True)
store_work = store_df.copy(deep=True)
submission_work = submission_df.copy(deep=True)

# Record original shapes for later validation
original_shapes = {
    "train": train_df.shape,
    "test": test_df.shape,
    "store": store_df.shape,
    "sample_submission": submission_df.shape
}

print("Original and working-copy shapes:")
for name, original_shape in original_shapes.items():
    working_shape = {
        "train": train_work.shape,
        "test": test_work.shape,
        "store": store_work.shape,
        "sample_submission": submission_work.shape
    }[name]

    print(
        f"{name}: original={original_shape}, "
        f"working={working_shape}"
    )

logger.info(
    "Working copies created; original source dataframes preserved."
)

2026-10-26 11:41:54 | INFO | Project6 | Working copies created; original source dataframes preserved.


Original and working-copy shapes:
train: original=(1017209, 9), working=(1017209, 9)
test: original=(41088, 8), working=(41088, 8)
store: original=(1115, 10), working=(1115, 10)
sample_submission: original=(41088, 2), working=(41088, 2)


### 8.2 **Data-Type and Schema Validation**

This section validates the column names and data types of the working datasets before any conversion or merging. Date columns will be checked for valid parsing, categorical fields will be inspected for consistency, and identifier and target columns will be verified. Changes will be made only when supported by the validation results.

In [216]:
# Validate working dataset schemas and data types

working_datasets = {
    "train": train_work,
    "test": test_work,
    "store": store_work,
    "sample_submission": submission_work
}

for name, df in working_datasets.items():
    print("\n" + "=" * 75)
    print(f"DATASET: {name.upper()}")
    print("=" * 75)

    print("Shape:", df.shape)
    print("Column names:", df.columns.tolist())

    schema_summary = pd.DataFrame({
        "Data Type": df.dtypes.astype(str),
        "Missing Count": df.isna().sum(),
        "Unique Values": df.nunique(dropna=True)
    })

    display(schema_summary)

    logger.info(
        "%s schema validated: %d rows and %d columns.",
        name,
        df.shape[0],
        df.shape[1]
    )


DATASET: TRAIN
Shape: (1017209, 9)
Column names: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


,Data Type,Missing Count,Unique Values
Store,int64,0,1115
DayOfWeek,int64,0,7
Date,str,0,942
Sales,int64,0,21734
Customers,int64,0,4086
Open,int64,0,2
Promo,int64,0,2
StateHoliday,string,0,4
SchoolHoliday,int64,0,2


2026-10-26 11:41:55 | INFO | Project6 | train schema validated: 1017209 rows and 9 columns.



DATASET: TEST
Shape: (41088, 8)
Column names: ['Id', 'Store', 'DayOfWeek', 'Date', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


,Data Type,Missing Count,Unique Values
Id,int64,0,41088
Store,int64,0,856
DayOfWeek,int64,0,7
Date,str,0,48
Open,float64,11,2
Promo,int64,0,2
StateHoliday,string,0,2
SchoolHoliday,int64,0,2


2026-10-26 11:41:55 | INFO | Project6 | test schema validated: 41088 rows and 8 columns.


... (2 outputs left out)

2026-10-26 11:41:55 | INFO | Project6 | store schema validated: 1115 rows and 10 columns.



DATASET: SAMPLE_SUBMISSION
Shape: (41088, 2)
Column names: ['Id', 'Sales']


,Data Type,Missing Count,Unique Values
Id,int64,0,41088
Sales,int64,0,1


2026-10-26 11:41:55 | INFO | Project6 | sample_submission schema validated: 41088 rows and 2 columns.


### 8.3 **Date and Categorical-Value Validation**

This section examines date values and categorical fields before standardization. It checks date parsing, holiday-category values, and binary indicator fields. The findings will guide subsequent preprocessing decisions without modifying the original source data.

In [217]:
# Inspect dates and categorical values
# No data is modified in this section.

# 1. Inspect date parsing in train and test
for name, df in [
    ("train", train_work),
    ("test", test_work)
]:
    parsed_dates = pd.to_datetime(
        df["Date"],
        errors="coerce"
    )

    print("\n" + "=" * 65)
    print(f"DATE INSPECTION: {name.upper()}")
    print("=" * 65)

    print("Original date dtype:", df["Date"].dtype)
    print("Unparseable dates:", parsed_dates.isna().sum())
    print("Earliest date:", parsed_dates.min())
    print("Latest date:", parsed_dates.max())
    print("Unique parsed dates:", parsed_dates.nunique())

    logger.info(
        "%s date inspection: unparseable=%d, unique_dates=%d",
        name,
        parsed_dates.isna().sum(),
        parsed_dates.nunique()
    )

# 2. Inspect holiday categories
for name, df in [
    ("train", train_work),
    ("test", test_work)
]:
    print(f"\nStateHoliday values in {name}:")
    print(df["StateHoliday"].value_counts(dropna=False))

    print(f"\nSchoolHoliday values in {name}:")
    print(df["SchoolHoliday"].value_counts(dropna=False))

# 3. Inspect binary indicators and missing values
for name, df in [
    ("train", train_work),
    ("test", test_work)
]:
    for column in ["Open", "Promo"]:
        print(
            f"\n{name} - {column} value counts:"
        )
        print(df[column].value_counts(dropna=False))

logger.info(
    "Date and categorical-value inspection completed."
)

2026-10-26 11:41:55 | INFO | Project6 | train date inspection: unparseable=0, unique_dates=942
2026-10-26 11:41:55 | INFO | Project6 | test date inspection: unparseable=0, unique_dates=48
2026-10-26 11:41:55 | INFO | Project6 | Date and categorical-value inspection completed.



DATE INSPECTION: TRAIN
Original date dtype: str
Unparseable dates: 0
Earliest date: 2013-01-01 00:00:00
Latest date: 2015-07-31 00:00:00
Unique parsed dates: 942

DATE INSPECTION: TEST
Original date dtype: str
Unparseable dates: 0
Earliest date: 2015-08-01 00:00:00
Latest date: 2015-09-17 00:00:00
Unique parsed dates: 48

StateHoliday values in train:
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: Int64

SchoolHoliday values in train:
SchoolHoliday
0    835488
1    181721
Name: count, dtype: int64

StateHoliday values in test:
StateHoliday
0    40908
a      180
Name: count, dtype: Int64

SchoolHoliday values in test:
SchoolHoliday
0    22866
1    18222
Name: count, dtype: int64

train - Open value counts:
Open
1    844392
0    172817
Name: count, dtype: int64

train - Promo value counts:
Promo
0    629129
1    388080
Name: count, dtype: int64

test - Open value counts:
Open
1.0    35093
0.0     5984
NaN       11
Name: count, dtype: int64

test - Promo

### 8.4 **Date Standardization and Categorical-Value Audit**

The date inspection found no unparseable dates in the training or test datasets. This section standardizes the date columns in the working copies and examines categorical values and binary indicators before further preprocessing. The original source dataframes remain unchanged.

In [218]:
# Standardize dates and inspect categorical values

# 1. Convert dates in working copies only
train_work["Date"] = pd.to_datetime(
    train_work["Date"],
    errors="raise"
)

test_work["Date"] = pd.to_datetime(
    test_work["Date"],
    errors="raise"
)

# 2. Confirm the conversion
print("TRAIN Date dtype:", train_work["Date"].dtype)
print("TEST Date dtype:", test_work["Date"].dtype)

print("\nTrain date range:")
print(train_work["Date"].min(), "to", train_work["Date"].max())

print("\nTest date range:")
print(test_work["Date"].min(), "to", test_work["Date"].max())

# 3. Show every category and its count
for name, df in [
    ("TRAIN", train_work),
    ("TEST", test_work)
]:
    print(f"\n{name}: StateHoliday categories")
    display(
        df["StateHoliday"]
        .value_counts(dropna=False)
        .rename_axis("Category")
        .reset_index(name="Count")
    )

    print(f"\n{name}: SchoolHoliday categories")
    display(
        df["SchoolHoliday"]
        .value_counts(dropna=False)
        .rename_axis("Category")
        .reset_index(name="Count")
    )

    print(f"\n{name}: Promo categories")
    display(
        df["Promo"]
        .value_counts(dropna=False)
        .rename_axis("Category")
        .reset_index(name="Count")
    )

logger.info(
    "Working-copy dates standardized and categorical values inspected."
)

TRAIN Date dtype: datetime64[us]
TEST Date dtype: datetime64[us]

Train date range:
2013-01-01 00:00:00 to 2015-07-31 00:00:00

Test date range:
2015-08-01 00:00:00 to 2015-09-17 00:00:00

TRAIN: StateHoliday categories


,Category,Count
0,0,986159
1,a,20260
2,b,6690
3,c,4100



TRAIN: SchoolHoliday categories


,Category,Count
0,0,835488
1,1,181721



TRAIN: Promo categories


,Category,Count
0,0,629129
1,1,388080


... (3 outputs left out)

,Category,Count
0,0,22866
1,1,18222



TEST: Promo categories


,Category,Count
0,0,24824
1,1,16264


2026-10-26 11:41:55 | INFO | Project6 | Working-copy dates standardized and categorical values inspected.


### 8.5 **Missing-Value Investigation and Treatment Strategy**

This section investigates missing values in the working copies before any imputation or row removal. Missingness is examined by column and, where relevant, by store metadata fields. Treatment decisions will be based on the meaning of each feature and the observed data rather than applying a blanket imputation rule.

The original source dataframes remain unchanged.

In [219]:
# Investigate missing values before treatment

datasets_to_check = {
    "TRAIN": train_work,
    "TEST": test_work,
    "STORE": store_work,
    "SAMPLE_SUBMISSION": submission_work
}

for name, df in datasets_to_check.items():
    print("\n" + "=" * 65)
    print(f"MISSING-VALUE INVESTIGATION: {name}")
    print("=" * 65)

    missing_report = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (
            df.isna().mean() * 100
        ).round(2)
    })

    missing_report = missing_report[
        missing_report["missing_count"] > 0
    ].sort_values(
        "missing_count",
        ascending=False
    )

    if missing_report.empty:
        print("No missing values found.")
    else:
        display(missing_report)

    logger.info(
        "%s missing-value investigation completed; "
        "%d columns contain missing values.",
        name,
        len(missing_report)
    )


MISSING-VALUE INVESTIGATION: TRAIN


2026-10-26 11:41:55 | INFO | Project6 | TRAIN missing-value investigation completed; 0 columns contain missing values.


No missing values found.

MISSING-VALUE INVESTIGATION: TEST


,data_type,missing_count,missing_percent
Open,float64,11,0.03


2026-10-26 11:41:55 | INFO | Project6 | TEST missing-value investigation completed; 1 columns contain missing values.



MISSING-VALUE INVESTIGATION: STORE


,data_type,missing_count,missing_percent
Promo2SinceYear,float64,544,48.79
Promo2SinceWeek,float64,544,48.79
PromoInterval,str,544,48.79
CompetitionOpenSinceMonth,float64,354,31.75
CompetitionOpenSinceYear,float64,354,31.75
CompetitionDistance,float64,3,0.27


2026-10-26 11:41:55 | INFO | Project6 | STORE missing-value investigation completed; 6 columns contain missing values.
2026-10-26 11:41:55 | INFO | Project6 | SAMPLE_SUBMISSION missing-value investigation completed; 0 columns contain missing values.



MISSING-VALUE INVESTIGATION: SAMPLE_SUBMISSION
No missing values found.


### 8.6 **Investigating Missing Store Metadata**

This section investigates missing values in store-level metadata by comparing the affected columns with related business indicators. In particular, Promo2-related missing values are examined against the Promo2 participation flag, while competition-related missing values are inspected separately.

No missing values will be imputed or removed until the investigation supports a defensible treatment decision.

In [220]:

# Investigate missing store metadata

# 1. Compare Promo2 participation with missing Promo2 fields
promo2_columns = [
    "Promo2SinceYear",
    "Promo2SinceWeek",
    "PromoInterval"
]

print("=" * 65)
print("PROMO2 PARTICIPATION AND MISSINGNESS")
print("=" * 65)

promo2_missing_report = pd.DataFrame({
    "Promo2": store_work["Promo2"],
    "Promo2SinceYear_missing": (
        store_work["Promo2SinceYear"].isna()
    ),
    "Promo2SinceWeek_missing": (
        store_work["Promo2SinceWeek"].isna()
    ),
    "PromoInterval_missing": (
        store_work["PromoInterval"].isna()
    )
})

display(
    promo2_missing_report.groupby("Promo2").agg(
        store_count=("Promo2", "size"),
        missing_since_year=("Promo2SinceYear_missing", "sum"),
        missing_since_week=("Promo2SinceWeek_missing", "sum"),
        missing_interval=("PromoInterval_missing", "sum")
    )
)

# 2. Investigate competition-related missing values
competition_columns = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear"
]

print("\n" + "=" * 65)
print("COMPETITION METADATA MISSINGNESS")
print("=" * 65)

competition_missing_report = pd.DataFrame({
    "missing_count": store_work[competition_columns].isna().sum(),
    "missing_percent": (
        store_work[competition_columns].isna().mean() * 100
    ).round(2)
})

display(competition_missing_report)

# 3. Display the specific affected store records
affected_mask = store_work[competition_columns].isna().any(axis=1)

print("\nStores with at least one missing competition field:")
display(
    store_work.loc[
        affected_mask,
        ["Store"] + competition_columns
    ].sort_values("Store")
)

logger.info(
    "Store metadata missingness investigation completed."
)


PROMO2 PARTICIPATION AND MISSINGNESS


,store_count,missing_since_year,missing_since_week,missing_interval
Promo2,,,,
0,544,544,544,544
1,571,0,0,0



COMPETITION METADATA MISSINGNESS


,missing_count,missing_percent
CompetitionDistance,3,0.27
CompetitionOpenSinceMonth,354,31.75
CompetitionOpenSinceYear,354,31.75



Stores with at least one missing competition field:


,Store,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
11,12,1070.0,NaN,NaN
12,13,310.0,NaN,NaN
15,16,3270.0,NaN,NaN
18,19,3240.0,NaN,NaN
21,22,1040.0,NaN,NaN
...,...,...,...,...
1095,1096,1130.0,NaN,NaN
1099,1100,540.0,NaN,NaN
1112,1113,9260.0,NaN,NaN
1113,1114,870.0,NaN,NaN


2026-10-26 11:41:55 | INFO | Project6 | Store metadata missingness investigation completed.


### 8.7 **Record-Level Investigation of Missing Values**

This section identifies the specific records affected by missing competition-distance and store-opening-status values. The purpose is to determine whether missing values can be explained by related dataset information before choosing an imputation or exclusion strategy.

No rows will be removed, and no missing values will be replaced at this stage.

In [221]:

# Inspect specific records with missing values

# 1. Identify stores with missing CompetitionDistance
print("=" * 65)
print("STORES WITH MISSING COMPETITION DISTANCE")
print("=" * 65)

missing_distance_stores = store_work.loc[
    store_work["CompetitionDistance"].isna(),
    ["Store", "StoreType", "Assortment", "CompetitionDistance",
     "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]
]

display(missing_distance_stores)

logger.info(
    "Identified %d stores with missing CompetitionDistance.",
    len(missing_distance_stores)
)

# 2. Inspect test rows where Open is missing
print("\n" + "=" * 65)
print("TEST RECORDS WITH MISSING OPEN STATUS")
print("=" * 65)

missing_open_test = test_work.loc[
    test_work["Open"].isna(),
    ["Id", "Store", "DayOfWeek", "Date", "Open",
     "Promo", "StateHoliday", "SchoolHoliday"]
]

display(missing_open_test)

logger.info(
    "Identified %d test rows with missing Open status.",
    len(missing_open_test)
)


STORES WITH MISSING COMPETITION DISTANCE


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear
290,291,d,a,NaN,NaN,NaN
621,622,a,c,NaN,NaN,NaN
878,879,d,a,NaN,NaN,NaN


2026-10-26 11:41:55 | INFO | Project6 | Identified 3 stores with missing CompetitionDistance.



TEST RECORDS WITH MISSING OPEN STATUS


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
479,480,622,4,2015-09-17,NaN,1,0,0
1335,1336,622,3,2015-09-16,NaN,1,0,0
2191,2192,622,2,2015-09-15,NaN,1,0,0
3047,3048,622,1,2015-09-14,NaN,1,0,0
4759,4760,622,6,2015-09-12,NaN,0,0,0
5615,5616,622,5,2015-09-11,NaN,0,0,0
6471,6472,622,4,2015-09-10,NaN,0,0,0
7327,7328,622,3,2015-09-09,NaN,0,0,0
8183,8184,622,2,2015-09-08,NaN,0,0,0
9039,9040,622,1,2015-09-07,NaN,0,0,0


2026-10-26 11:41:55 | INFO | Project6 | Identified 11 test rows with missing Open status.



### 8.8 **Missing-Value Investigation and Treatment Strategy**

The source-data audit identified 11 missing `Open` values in the test dataset and missing values in selected store metadata columns.

Treatment decisions:
1. **Test `Open`:** Preserve the 11 missing values during this preprocessing stage. Historical opening patterns are evidence only and do not confirm the actual missing values.
2. **CompetitionDistance:** Preserve the three missing values and record a missingness indicator. Any numerical imputation will be selected and fitted using training data within the modeling pipeline.
3. **Competition opening month/year:** Preserve missing values because the exact opening dates are unknown.
4. **Promo2-related metadata:** Missing values for stores with `Promo2 = 0` are consistent with non-participation. Preserve the original missing values at this stage rather than filling them indiscriminately.
5. **Original data preservation:** Apply all transformations to working copies. Do not delete records or change source DataFrames.

No rows will be removed solely because of missing values. Missingness will be re-audited after store metadata is merged.


In [222]:

# Investigate Store 622 before treating missing values

# 1. Inspect Store 622 metadata
print("=" * 65)
print("STORE 622: STORE METADATA")
print("=" * 65)

store_622_metadata = store_work.loc[
    store_work["Store"].eq(622)
]

display(store_622_metadata)

# 2. Inspect all available training records for Store 622
print("\n" + "=" * 65)
print("STORE 622: TRAINING RECORDS")
print("=" * 65)

store_622_train = train_work.loc[
    train_work["Store"].eq(622)
].sort_values("Date")

display(store_622_train.tail(15))

# 3. Inspect all test records for Store 622
print("\n" + "=" * 65)
print("STORE 622: TEST RECORDS")
print("=" * 65)

store_622_test = test_work.loc[
    test_work["Store"].eq(622)
].sort_values("Date")

display(store_622_test)

# 4. Log the investigation
logger.info(
    "Store 622 investigation completed: training rows=%d, test rows=%d.",
    len(store_622_train),
    len(store_622_test)
)


STORE 622: STORE METADATA


,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
621,622,a,c,NaN,NaN,NaN,0,NaN,NaN,NaN



STORE 622: TRAINING RECORDS


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
16231,622,5,2015-07-17,5263,462,1,1,0,0
15116,622,6,2015-07-18,2367,265,1,0,0,0
14001,622,7,2015-07-19,0,0,0,0,0,0
12886,622,1,2015-07-20,4688,429,1,0,0,1
11771,622,2,2015-07-21,3529,367,1,0,0,1
10656,622,3,2015-07-22,3721,376,1,0,0,1
9541,622,4,2015-07-23,3970,399,1,0,0,1
8426,622,5,2015-07-24,4183,440,1,0,0,1
7311,622,6,2015-07-25,2644,257,1,0,0,0
6196,622,7,2015-07-26,0,0,0,0,0,0



STORE 622: TEST RECORDS


,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
40711,40712,622,6,2015-08-01,1.0,0,0,0
39855,39856,622,7,2015-08-02,0.0,0,0,0
38999,39000,622,1,2015-08-03,1.0,1,0,1
38143,38144,622,2,2015-08-04,1.0,1,0,1
37287,37288,622,3,2015-08-05,1.0,1,0,1
36431,36432,622,4,2015-08-06,1.0,1,0,1
35575,35576,622,5,2015-08-07,1.0,1,0,1
34719,34720,622,6,2015-08-08,1.0,0,0,0
33863,33864,622,7,2015-08-09,0.0,0,0,0
33007,33008,622,1,2015-08-10,1.0,0,0,1


2026-10-26 11:41:55 | INFO | Project6 | Store 622 investigation completed: training rows=942, test rows=48.


### 8.8.1 **Historical Open-Status Pattern for Store 622**

The missing `Open` values will be investigated using the store's historical operating status. The analysis will compare opening patterns by weekday and promotion status using the training data, then examine the known and missing statuses in the test period.

No missing values will be imputed until the evidence and limitations have been documented.

In [223]:

# Analyse historical opening patterns for Store 622

# Work only with Store 622's training records.
# The original source datasets remain unchanged.

historical_622 = train_work.loc[
    train_work["Store"].eq(622)
].copy()

test_622 = test_work.loc[
    test_work["Store"].eq(622)
].copy()

# Convert dates in these working subsets for reliable sorting.
historical_622["Date"] = pd.to_datetime(
    historical_622["Date"], errors="coerce"
)

test_622["Date"] = pd.to_datetime(
    test_622["Date"], errors="coerce"
)

# 1. Historical opening behaviour by weekday.
print("=" * 70)
print("STORE 622: HISTORICAL OPENING RATE BY DAY OF WEEK")
print("=" * 70)

weekday_opening_pattern = (
    historical_622
    .groupby("DayOfWeek", dropna=False)
    .agg(
        records=("Open", "size"),
        known_open_values=("Open", "count"),
        open_days=("Open", lambda s: s.eq(1).sum()),
        closed_days=("Open", lambda s: s.eq(0).sum()),
        opening_rate=("Open", "mean")
    )
    .reset_index()
)

display(weekday_opening_pattern)

# 2. Historical opening behaviour by weekday and promotion.
print("\n" + "=" * 70)
print("STORE 622: OPENING PATTERN BY WEEKDAY AND PROMO")
print("=" * 70)

weekday_promo_pattern = (
    historical_622
    .groupby(["DayOfWeek", "Promo"], dropna=False)
    .agg(
        records=("Open", "size"),
        open_days=("Open", lambda s: s.eq(1).sum()),
        closed_days=("Open", lambda s: s.eq(0).sum()),
        opening_rate=("Open", "mean")
    )
    .reset_index()
)

display(weekday_promo_pattern)

# 3. Inspect known and missing Open statuses in the test period.
print("\n" + "=" * 70)
print("STORE 622: TEST OPEN-STATUS AUDIT")
print("=" * 70)

test_open_audit = test_622[
    ["Id", "Store", "Date", "DayOfWeek", "Promo", "Open"]
].sort_values("Date")

display(test_open_audit)

# 4. Log the investigation summary.
logger.info(
    "Store 622 opening-pattern analysis completed. "
    "Historical rows=%d; test rows=%d; missing test Open=%d.",
    len(historical_622),
    len(test_622),
    test_622["Open"].isna().sum()
)


STORE 622: HISTORICAL OPENING RATE BY DAY OF WEEK


,DayOfWeek,records,known_open_values,open_days,closed_days,opening_rate
0,1,134,134,128,6,0.955224
1,2,135,135,134,1,0.992593
2,3,135,135,132,3,0.977778
3,4,135,135,127,8,0.940741
4,5,135,135,129,6,0.955556
5,6,134,134,134,0,1.000000
6,7,134,134,0,134,0.000000



STORE 622: OPENING PATTERN BY WEEKDAY AND PROMO


,DayOfWeek,Promo,records,open_days,closed_days,opening_rate
0,1,0,62,56,6,0.903226
1,1,1,72,72,0,1.000000
2,2,0,63,62,1,0.984127
3,2,1,72,72,0,1.000000
4,3,0,63,61,2,0.968254
5,3,1,72,71,1,0.986111
6,4,0,63,56,7,0.888889
7,4,1,72,71,1,0.986111
8,5,0,63,62,1,0.984127
9,5,1,72,67,5,0.930556



STORE 622: TEST OPEN-STATUS AUDIT


,Id,Store,Date,DayOfWeek,Promo,Open
40711,40712,622,2015-08-01,6,0,1.0
39855,39856,622,2015-08-02,7,0,0.0
38999,39000,622,2015-08-03,1,1,1.0
38143,38144,622,2015-08-04,2,1,1.0
37287,37288,622,2015-08-05,3,1,1.0
36431,36432,622,2015-08-06,4,1,1.0
35575,35576,622,2015-08-07,5,1,1.0
34719,34720,622,2015-08-08,6,0,1.0
33863,33864,622,2015-08-09,7,0,0.0
33007,33008,622,2015-08-10,1,0,1.0


2026-10-26 11:41:55 | INFO | Project6 | Store 622 opening-pattern analysis completed. Historical rows=942; test rows=48; missing test Open=11.



### 8.8.2 — **Investigation of Missing Open Values**

**Objective:** Investigate missing `Open` values in the test dataset before deciding how to handle them.

**Methodology:**
- Review missing `Open` records by store, date, weekday, and promotion status.
- Compare them with historical opening patterns.
- Check whether the available evidence supports a reliable treatment decision.
- Preserve the original data and document any assumptions before applying changes.

**Important:** Missing values will not be automatically replaced with zero or one without supporting evidence. Historical patterns are evidence, not confirmation of the actual missing status.


In [224]:
# Investigate missing Open values
# Purpose: Review missing records without modifying the original data.

import pandas as pd

# Confirm that the required working DataFrames exist.
required_objects = ["train_work", "test_work", "store_work"]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Required DataFrames not found: {missing_objects}. "
        "Check the earlier preprocessing cells before continuing."
    )

# Confirm that the required columns exist.
required_test_columns = [
    "Store", "Date", "DayOfWeek", "Open", "Promo"
]

missing_columns = [
    col for col in required_test_columns
    if col not in test_work.columns
]

if missing_columns:
    raise KeyError(
        f"Required columns missing from test_work: {missing_columns}"
    )

# Select only records where Open is missing.
missing_open = test_work.loc[
    test_work["Open"].isna(),
    required_test_columns
].copy()

# Standardize dates in this temporary inspection copy only.
missing_open["Date"] = pd.to_datetime(
    missing_open["Date"],
    errors="coerce"
)

print("MISSING OPEN VALUE SUMMARY")
print("=" * 60)
print(f"Total missing Open records: {len(missing_open)}")
print(f"Stores affected: {missing_open['Store'].nunique()}")
print(f"Unparseable dates: {missing_open['Date'].isna().sum()}")

print("\nMissing records by store:")
print(
    missing_open.groupby("Store", dropna=False)
    .size()
    .sort_values(ascending=False)
)

print("\nMissing records by weekday:")
print(
    missing_open.groupby("DayOfWeek", dropna=False)
    .size()
    .sort_index()
)

print("\nMissing records by promotion status:")
print(
    missing_open.groupby("Promo", dropna=False)
    .size()
    .sort_index()
)

print("\nDetailed missing records:")
print(
    missing_open.sort_values(["Store", "Date"])
    .to_string(index=False)
)

# Log the investigation if the project's logger is available.
if "logger" in globals():
    logger.info(
        "Missing Open investigation completed: %s records across %s stores.",
        len(missing_open),
        missing_open["Store"].nunique()
    )

2026-10-26 11:41:55 | INFO | Project6 | Missing Open investigation completed: 11 records across 1 stores.


MISSING OPEN VALUE SUMMARY
Total missing Open records: 11
Stores affected: 1
Unparseable dates: 0

Missing records by store:
Store
622    11
dtype: int64

Missing records by weekday:
DayOfWeek
1    2
2    2
3    2
4    2
5    1
6    2
dtype: int64

Missing records by promotion status:
Promo
0    7
1    4
dtype: int64

Detailed missing records:
 Store       Date  DayOfWeek  Open  Promo
   622 2015-09-05          6   NaN      0
   622 2015-09-07          1   NaN      0
   622 2015-09-08          2   NaN      0
   622 2015-09-09          3   NaN      0
   622 2015-09-10          4   NaN      0
   622 2015-09-11          5   NaN      0
   622 2015-09-12          6   NaN      0
   622 2015-09-14          1   NaN      1
   622 2015-09-15          2   NaN      1
   622 2015-09-16          3   NaN      1
   622 2015-09-17          4   NaN      1


### 8.8.3 — **Evidence-Based Validation of Missing Open Values**

**Objective:** Compare the missing `Open` records with the same store's historical records for matching weekdays and promotion conditions.

**Validation rules:**
- Check historical opening and closing patterns.
- Compare matching weekdays and promotion status.
- Keep uncertain cases unresolved until sufficient evidence is available.
- Do not modify the original test dataset during investigation.

In [225]:
# Validate missing Open values using historical evidence

import pandas as pd

# Work on copies so the original datasets remain unchanged.
train_check = train_work.copy()
test_check = test_work.copy()

train_check["Date"] = pd.to_datetime(
    train_check["Date"], errors="coerce"
)
test_check["Date"] = pd.to_datetime(
    test_check["Date"], errors="coerce"
)

# Identify test records where Open is missing.
missing_records = test_check.loc[
    test_check["Open"].isna(),
    ["Id", "Store", "Date", "DayOfWeek", "Promo"]
].copy()

results = []

for _, row in missing_records.iterrows():

    # Historical records for the same store, weekday and Promo status.
    history = train_check.loc[
        (train_check["Store"] == row["Store"])
        & (train_check["DayOfWeek"] == row["DayOfWeek"])
        & (train_check["Promo"] == row["Promo"])
        & (train_check["Date"] < row["Date"])
        & (train_check["Open"].notna())
    ]

    # Summarize available historical evidence.
    known_values = history["Open"].value_counts().to_dict()
    total_known = int(history["Open"].notna().sum())

    results.append({
        "Id": row["Id"],
        "Store": row["Store"],
        "Date": row["Date"],
        "DayOfWeek": row["DayOfWeek"],
        "Promo": row["Promo"],
        "historical_records": total_known,
        "historical_open_0": int(known_values.get(0, 0)),
        "historical_open_1": int(known_values.get(1, 0))
    })

evidence_df = pd.DataFrame(results)

print("HISTORICAL EVIDENCE FOR MISSING OPEN VALUES")
print("=" * 65)
print(evidence_df.to_string(index=False))

print("\nInvestigation summary:")
print(f"Records checked: {len(evidence_df)}")
print(
    "Records with no matching historical evidence:",
    int((evidence_df["historical_records"] == 0).sum())
)

# Log the result if the project logger exists.
if "logger" in globals():
    logger.info(
        "Historical evidence check completed for %s missing Open records.",
        len(evidence_df)
    )

HISTORICAL EVIDENCE FOR MISSING OPEN VALUES
   Id  Store       Date  DayOfWeek  Promo  historical_records  historical_open_0  historical_open_1
  480    622 2015-09-17          4      1                  72                  1                 71
 1336    622 2015-09-16          3      1                  72                  1                 71
 2192    622 2015-09-15          2      1                  72                  0                 72
 3048    622 2015-09-14          1      1                  72                  0                 72
 4760    622 2015-09-12          6      0                 134                  0                134
 5616    622 2015-09-11          5      0                  63                  1                 62
 6472    622 2015-09-10          4      0                  63                  7                 56
 7328    622 2015-09-09          3      0                  63                  2                 61
 8184    622 2015-09-08          2      0               

2026-10-26 11:41:55 | INFO | Project6 | Historical evidence check completed for 11 missing Open records.


In [226]:

# 8.8.4 — Summarize confidence in historical opening patterns
# This cell investigates the evidence only.
# It does not change the original train or test datasets.

# Calculate historical opening and closing rates.
evidence_df["historical_open_rate"] = (
    evidence_df["historical_open_1"]
    / evidence_df["historical_records"].replace(0, pd.NA)
)

evidence_df["historical_closed_rate"] = (
    evidence_df["historical_open_0"]
    / evidence_df["historical_records"].replace(0, pd.NA)
)


# Classify historical consistency, not the actual missing value.
def classify_evidence(row):
    if row["historical_records"] == 0:
        return "Insufficient evidence"

    if row["historical_open_rate"] >= 0.95:
        return "Historically usually open"

    if row["historical_closed_rate"] >= 0.95:
        return "Historically usually closed"

    return "Mixed historical pattern"


evidence_df["evidence_assessment"] = evidence_df.apply(
    classify_evidence,
    axis=1
)


# Prepare a display-only copy to avoid changing the original evidence.
evidence_display = evidence_df[
    [
        "Id",
        "Date",
        "DayOfWeek",
        "Promo",
        "historical_records",
        "historical_open_rate",
        "historical_closed_rate",
        "evidence_assessment",
    ]
].copy()


# Format dates safely for display.
evidence_display["Date"] = pd.to_datetime(
    evidence_display["Date"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")


# Round only the numeric rate columns.
rate_columns = [
    "historical_open_rate",
    "historical_closed_rate",
]

evidence_display[rate_columns] = (
    evidence_display[rate_columns].round(3)
)


# Display the historical evidence summary.
display(evidence_display)


,Id,Date,DayOfWeek,Promo,historical_records,historical_open_rate,historical_closed_rate,evidence_assessment
0,480,2015-09-17,4,1,72,0.986,0.014,Historically usually open
1,1336,2015-09-16,3,1,72,0.986,0.014,Historically usually open
2,2192,2015-09-15,2,1,72,1.000,0.000,Historically usually open
3,3048,2015-09-14,1,1,72,1.000,0.000,Historically usually open
4,4760,2015-09-12,6,0,134,1.000,0.000,Historically usually open
5,5616,2015-09-11,5,0,63,0.984,0.016,Historically usually open
6,6472,2015-09-10,4,0,63,0.889,0.111,Mixed historical pattern
7,7328,2015-09-09,3,0,63,0.968,0.032,Historically usually open
8,8184,2015-09-08,2,0,63,0.984,0.016,Historically usually open
9,9040,2015-09-07,1,0,62,0.903,0.097,Mixed historical pattern



## 8.9 **Prepare Working Data for Store Metadata Merging**

Before merging, confirm that the working DataFrames contain the expected keys and that the store metadata has one row per store.

The source DataFrames remain unchanged. Missing metadata values will be preserved during merging and reviewed afterward. Numerical imputation and model-specific preprocessing will be handled later without using test-target information.


In [227]:

# Validate merge prerequisites

# Confirm that the required working DataFrames exist.
required_objects = ["train_work", "test_work", "store_work"]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing working DataFrames: {missing_objects}. "
        "Run the earlier preprocessing cells first."
    )

# Confirm the Store key exists in every DataFrame.
for name, df in {
    "train_work": train_work,
    "test_work": test_work,
    "store_work": store_work
}.items():
    if "Store" not in df.columns:
        raise KeyError(f"'Store' column missing from {name}.")

# The metadata must contain one record per store.
duplicate_metadata_keys = int(
    store_work["Store"].duplicated().sum()
)

if duplicate_metadata_keys > 0:
    raise ValueError(
        f"store_work contains {duplicate_metadata_keys} duplicate Store keys."
    )

# Check for unmatched store IDs before merging.
train_unmatched = set(train_work["Store"]) - set(store_work["Store"])
test_unmatched = set(test_work["Store"]) - set(store_work["Store"])

print("Train rows before merge:", len(train_work))
print("Test rows before merge:", len(test_work))
print("Store metadata rows:", len(store_work))
print("Duplicate Store keys in metadata:", duplicate_metadata_keys)
print("Train store IDs missing from metadata:", len(train_unmatched))
print("Test store IDs missing from metadata:", len(test_unmatched))

if train_unmatched or test_unmatched:
    raise ValueError(
        "Some train/test stores have no matching metadata. "
        "Investigate these IDs before merging."
    )

logger.info("Merge prerequisites validated successfully.")


Train rows before merge: 1017209
Test rows before merge: 41088
Store metadata rows: 1115
Duplicate Store keys in metadata: 0
Train store IDs missing from metadata: 0
Test store IDs missing from metadata: 0


2026-10-26 11:41:55 | INFO | Project6 | Merge prerequisites validated successfully.



### 8.10 **Merge Store Metadata with Training Data**

Merge the training records with the store metadata using a left join on `Store`. The left join preserves all training records, while `validate="many_to_one"` prevents duplicate metadata keys from multiplying rows.


In [228]:

#  Merge training data with store metadata

train_merge_check = train_work.merge(
    store_work,
    on="Store",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Training rows before merge:", len(train_work))
print("Training rows after merge:", len(train_merge_check))

print("\nTraining merge status:")
display(
    train_merge_check["_merge"]
    .value_counts(dropna=False)
    .rename_axis("Merge Status")
    .reset_index(name="Row Count")
)

# Fail early if any training row lacks store metadata.
if len(train_merge_check) != len(train_work):
    raise ValueError("Training row count changed during metadata merge.")

if not train_merge_check["_merge"].eq("both").all():
    raise ValueError("Some training rows did not match store metadata.")

# Remove the diagnostic merge indicator after successful validation.
train_merged = train_merge_check.drop(columns="_merge").copy()

logger.info(
    "Training metadata merge completed: %d rows, %d columns.",
    train_merged.shape[0],
    train_merged.shape[1]
)

print("\nFinal training merged shape:", train_merged.shape)


Training rows before merge: 1017209
Training rows after merge: 1017209

Training merge status:


,Merge Status,Row Count
0,both,1017209
1,left_only,0
2,right_only,0


2026-10-26 11:41:56 | INFO | Project6 | Training metadata merge completed: 1017209 rows, 18 columns.



Final training merged shape: (1017209, 18)



### 8.11 **Merge Store Metadata with Test Data**

Merge the test records with store metadata using a left join on `Store`. Preserve all test IDs and the original test row order. The test dataset does not contain observed sales targets, so no sales labels will be created or inferred during this step.


In [229]:

# Merge test data with store metadata

# Preserve the original test ID order for validation.
test_ids_before_merge = test_work["Id"].copy()

test_merge_check = test_work.merge(
    store_work,
    on="Store",
    how="left",
    validate="many_to_one",
    indicator=True,
    sort=False
)

print("Test rows before merge:", len(test_work))
print("Test rows after merge:", len(test_merge_check))

print("\nTest merge status:")
display(
    test_merge_check["_merge"]
    .value_counts(dropna=False)
    .rename_axis("Merge Status")
    .reset_index(name="Row Count")
)

# Validate row count, metadata matching, and ID order.
if len(test_merge_check) != len(test_work):
    raise ValueError("Test row count changed during metadata merge.")

if not test_merge_check["_merge"].eq("both").all():
    raise ValueError("Some test rows did not match store metadata.")

if not test_merge_check["Id"].reset_index(drop=True).equals(
    test_ids_before_merge.reset_index(drop=True)
):
    raise ValueError("Test ID order changed during the merge.")

# Remove the diagnostic merge indicator after validation.
test_merged = test_merge_check.drop(columns="_merge").copy()

logger.info(
    "Test metadata merge completed: %d rows, %d columns.",
    test_merged.shape[0],
    test_merged.shape[1]
)

print("\nFinal test merged shape:", test_merged.shape)
print(
    "Test ID order preserved:",
    test_merged["Id"].reset_index(drop=True).equals(
        test_ids_before_merge.reset_index(drop=True)
    )
)


Test rows before merge: 41088
Test rows after merge: 41088

Test merge status:


,Merge Status,Row Count
0,both,41088
1,left_only,0
2,right_only,0


2026-10-26 11:41:56 | INFO | Project6 | Test metadata merge completed: 41088 rows, 17 columns.



Final test merged shape: (41088, 17)
Test ID order preserved: True



### 8.12 **Post-Merge Data Integrity and Missing-Value Validation**

Validate the merged training and test datasets after store metadata integration. Check row counts, duplicate keys, store matching, required columns, target availability, and remaining missing values.

Missing values are reported rather than automatically filled. Their treatment will be handled in the relevant modeling pipeline using documented, training-only rules.


In [230]:

#  Validate merged datasets


# Create a missingness indicator for CompetitionDistance.
# Preserve the original missing distance values.

for df in [train_merged, test_merged]:
    df["CompetitionDistance_missing"] = (
        df["CompetitionDistance"].isna().astype("int8")
    )

print(
    "Train missing CompetitionDistance indicators:",
    int(train_merged["CompetitionDistance_missing"].sum())
)

print(
    "Test missing CompetitionDistance indicators:",
    int(test_merged["CompetitionDistance_missing"].sum())
)


post_merge_checks = []

def add_check(check_name, result, expected):
    post_merge_checks.append({
        "Check": check_name,
        "Result": result,
        "Expected": expected,
        "Passed": result == expected
    })

# Row-count integrity
add_check(
    "Training row count preserved",
    len(train_merged),
    len(train_work)
)
add_check(
    "Test row count preserved",
    len(test_merged),
    len(test_work)
)

# Store-Date key integrity
add_check(
    "Duplicate training Store-Date pairs",
    int(train_merged.duplicated(["Store", "Date"]).sum()),
    0
)
add_check(
    "Duplicate test Store-Date pairs",
    int(test_merged.duplicated(["Store", "Date"]).sum()),
    0
)

# Store metadata match
add_check(
    "Training rows with missing StoreType after merge",
    int(train_merged["StoreType"].isna().sum()),
    0
)
add_check(
    "Test rows with missing StoreType after merge",
    int(test_merged["StoreType"].isna().sum()),
    0
)

# Target and ID integrity
add_check(
    "Training Sales column present",
    "Sales" in train_merged.columns,
    True
)
add_check(
    "Test Sales column absent",
    "Sales" not in test_merged.columns,
    True
)
add_check(
    "Test IDs preserved in original order",
    bool(
        test_merged["Id"].reset_index(drop=True).equals(
            test_work["Id"].reset_index(drop=True)
        )
    ),
    True
)

post_merge_summary = pd.DataFrame(post_merge_checks)
display(post_merge_summary)

if not post_merge_summary["Passed"].all():
    logger.error("One or more post-merge validation checks failed.")
    display(
        post_merge_summary.loc[~post_merge_summary["Passed"]]
    )
    raise ValueError(
        "Post-merge validation failed. Review the failed checks."
    )

logger.info("All post-merge integrity checks passed.")
print("PASS: All post-merge integrity checks passed.")

# Report missing values without imputing them.
print("\nMissing values in merged training data:")
display(
    train_merged.isna().sum()
    .loc[lambda s: s > 0]
    .rename("Missing Count")
    .to_frame()
)

print("\nMissing values in merged test data:")
display(
    test_merged.isna().sum()
    .loc[lambda s: s > 0]
    .rename("Missing Count")
    .to_frame()
)


Train missing CompetitionDistance indicators: 2642
Test missing CompetitionDistance indicators: 96


,Check,Result,Expected,Passed
0,Training row count preserved,1017209,1017209,True
1,Test row count preserved,41088,41088,True
2,Duplicate training Store-Date pairs,0,0,True
3,Duplicate test Store-Date pairs,0,0,True
4,Training rows with missing StoreType after merge,0,0,True
5,Test rows with missing StoreType after merge,0,0,True
6,Training Sales column present,True,True,True
7,Test Sales column absent,True,True,True
8,Test IDs preserved in original order,True,True,True


2026-10-26 11:41:56 | INFO | Project6 | All post-merge integrity checks passed.


PASS: All post-merge integrity checks passed.

Missing values in merged training data:


,Missing Count
CompetitionDistance,2642
CompetitionOpenSinceMonth,323348
CompetitionOpenSinceYear,323348
Promo2SinceWeek,508031
Promo2SinceYear,508031
PromoInterval,508031



Missing values in merged test data:


,Missing Count
Open,11
CompetitionDistance,96
CompetitionOpenSinceMonth,15216
CompetitionOpenSinceYear,15216
Promo2SinceWeek,17232
Promo2SinceYear,17232
PromoInterval,17232



### 8.13 **Final Merged Dataset Summary and Export**

This section summarizes the final merged datasets and saves them to the processed-data directory for use in subsequent project notebooks.

The original source CSV files remain unchanged. The exported files preserve the remaining missing values; no unsupported imputation has been applied.


In [231]:

#  Summarize and export final merged datasets

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Summarize final merged datasets.
final_dataset_summary = pd.DataFrame([
    {
        "Dataset": "train_merged",
        "Rows": train_merged.shape[0],
        "Columns": train_merged.shape[1],
        "Missing Cells": int(train_merged.isna().sum().sum()),
        "Duplicate Rows": int(train_merged.duplicated().sum())
    },
    {
        "Dataset": "test_merged",
        "Rows": test_merged.shape[0],
        "Columns": test_merged.shape[1],
        "Missing Cells": int(test_merged.isna().sum().sum()),
        "Duplicate Rows": int(test_merged.duplicated().sum())
    }
])

display(final_dataset_summary)

# Export merged datasets without DataFrame index columns.
train_output_path = PROCESSED_DIR / "train_merged.csv"
test_output_path = PROCESSED_DIR / "test_merged.csv"

train_merged.to_csv(train_output_path, index=False)
test_merged.to_csv(test_output_path, index=False)

print("Processed-data directory:", PROCESSED_DIR)
print("Training dataset saved to:", train_output_path)
print("Test dataset saved to:", test_output_path)

# Verify that the exported files exist.
if not train_output_path.exists() or not test_output_path.exists():
    raise FileNotFoundError(
        "One or both merged CSV files could not be verified."
    )

logger.info(
    "Final merged datasets exported successfully to %s",
    PROCESSED_DIR
)


,Dataset,Rows,Columns,Missing Cells,Duplicate Rows
0,train_merged,1017209,19,2173431,0
1,test_merged,41088,18,82235,0


2026-10-26 11:42:00 | INFO | Project6 | Final merged datasets exported successfully to d:\Project_6_Rossmann_Forecasting\data\processed


Processed-data directory: d:\Project_6_Rossmann_Forecasting\data\processed
Training dataset saved to: d:\Project_6_Rossmann_Forecasting\data\processed\train_merged.csv
Test dataset saved to: d:\Project_6_Rossmann_Forecasting\data\processed\test_merged.csv
